# Step 4 — Model B phase 1: projector alignment (thin notebook: it only calls scripts)
**Settings:** GPU T4 x2, Internet on, Secret `HF_TOKEN` ticked. Your HF account must have accepted the terms of
**google/medsiglip-448** and **google/gemma-3-1b-it** (open each page once, click *Agree*).
**Inputs:** Indiana, ReXGradient test images + train subset (notebook outputs), `ashery/chexpert`,
`organizations/nih-chest-xrays/data`, and the output of your finished Step 0/1 notebook (for `data/`).

Model B = frozen MedSigLIP-448 -> projector (2x2 patch merge -> Linear-GELU-Linear) -> frozen Gemma 3 1B.
Only the projector trains here, on real train reports + template reports from CheXpert/NIH labels.
Accept: loss converges and val reports are coherent (rules fixed in `configs/step4.yaml` `acceptance`).

In [ ]:
!git clone -q --depth 1 https://github.com/Vinith-44/Radioreport /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["RUN_NOTE"] = "step4 kaggle"

## 0. Restore earlier outputs (data/ parquet; a previous Step 4 session to resume)

In [ ]:
PREV = []   # e.g. ["/kaggle/input/step1-kaggle/project"] (+ a previous step4 output to resume)
os.makedirs("/kaggle/working/project", exist_ok=True)
for src in PREV:
    !cp -rn {src}/outputs /kaggle/working/project/ 2>/dev/null || true
    !cp -rn {src}/data /kaggle/working/project/ 2>/dev/null || true
!python scripts/step0_prepare_data.py --config configs/step4.yaml
!python -m pytest tests -q

## 1. Smoke run (48 examples, 6 steps): read the sanity-check line and the generated samples

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step4.yaml --smoke
!python scripts/step4_train_projector.py --run-name b1 --smoke

## 2. Full run on both T4s (data parallel). Resumable: after a timeout, restore this output and re-run.
Prints s/step and the ETA after the first steps; if it is too slow, lower `align_data` sizes in the config.

In [ ]:
!torchrun --nproc_per_node 2 scripts/step4_train_projector.py --run-name b1
!cat /kaggle/working/project/outputs/runs/step4/b1/acceptance.json
!head -c 3000 /kaggle/working/project/outputs/runs/step4/b1/val_samples.jsonl